In [ ]:
import numpy as np

# Fungsi aktivasi bipolar
def activation_bipolar(x):
    return 1 if x >= 0 else -1

# Dataset fungsi OR dalam bentuk bipolar
data = np.array([
    [-1, -1, -1],
    [-1,  1,  1],
    [ 1, -1,  1],
    [ 1,  1,  1],
])

# Inisialisasi bobot dan bias
weights = np.zeros(2, dtype=float)
bias = 0.0
learning_rate = 1
epochs = 10

# Training
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}")
    print(" X1  X2  Target   Yin   Yout  Error   Δw1   Δw2   Δb   w1    w2    b")
    error_count = 0
    for x1, x2, target in data:
        x = np.array([x1, x2])
        yin = np.dot(weights, x) + bias
        yout = activation_bipolar(yin)
        error = target - yout

        # Delta
        delta_w = learning_rate * error * x
        delta_b = learning_rate * error

        # Update bobot jika ada error
        if error != 0:
            weights += delta_w
            bias += delta_b
            error_count += 1

        print(f"{x1:>3} {x2:>3} {target:>7} {yin:>7.2f} {yout:>6} {error:>7} "
              f"{delta_w[0]:>6.1f} {delta_w[1]:>6.1f} {delta_b:>6.1f} "
              f"{weights[0]:>5.1f} {weights[1]:>5.1f} {bias:>5.1f}")

    if error_count == 0:
        print("Training selesai karena tidak ada error.\n")
        break
    print()


Epoch 1
 X1  X2  Target   Yin   Yout  Error   Δw1   Δw2   Δb   w1    w2    b
 -1  -1      -1    0.00      1      -2    2.0    2.0   -2.0   2.0   2.0  -2.0
 -1   1       1   -2.00     -1       2   -2.0    2.0    2.0   0.0   4.0   0.0
  1  -1       1   -4.00     -1       2    2.0   -2.0    2.0   2.0   2.0   2.0
  1   1       1    6.00      1       0    0.0    0.0    0.0   2.0   2.0   2.0

Epoch 2
 X1  X2  Target   Yin   Yout  Error   Δw1   Δw2   Δb   w1    w2    b
 -1  -1      -1   -2.00     -1       0    0.0    0.0    0.0   2.0   2.0   2.0
 -1   1       1    2.00      1       0    0.0    0.0    0.0   2.0   2.0   2.0
  1  -1       1    2.00      1       0    0.0    0.0    0.0   2.0   2.0   2.0
  1   1       1    6.00      1       0    0.0    0.0    0.0   2.0   2.0   2.0
Training selesai karena tidak ada error.



In [19]:
import numpy as np
import pandas as pd

class Perceptron:
    def __init__(self, num_features, alpha=1.0, theta=0.5):
        self.weights = np.zeros(num_features)
        self.bias = 0.0
        self.alpha = alpha
        self.theta = theta

    def step_function(self, x):
        return 1 if x >= self.theta else -1

    def predict(self, x):
        total = np.dot(x, self.weights) + self.bias
        return self.step_function(total)

    def fit(self, X, y, max_epochs=100):
        epoch = 1
        feature_headers = [f"X{i+1}" for i in range(X.shape[1])]

        while epoch <= max_epochs:
            print(f"\nEpoch {epoch}")
            header = (
                "".join([f"{h:>5}" for h in feature_headers]) +
                f"{'Yin':>10}{'Yout':>8}{'Target':>8}{'Error':>8}{'Δb':>6}" +
                "".join([f"{'Δw'+str(i+1):>8}" for i in range(X.shape[1])]) +
                "".join([f"{'w'+str(i+1):>8}" for i in range(X.shape[1])]) +
                f"{'Bias':>8}"
            )
            print(header)
            print("-" * len(header))

            total_changes = 0

            for xi, yi in zip(X, y):
                yin = np.dot(xi, self.weights) + self.bias
                yout = self.step_function(yin)
                error = yi - yout

                delta_w = self.alpha * error * xi
                delta_b = self.alpha * error

                if error != 0:
                    self.weights += delta_w
                    self.bias += delta_b
                    total_changes += 1

                row = (
                    "".join([f"{val:5.0f}" for val in xi]) +
                    f"{yin:10.2f}{yout:8}{yi:8}{error:8}{delta_b:6.2f}" +
                    "".join([f"{dw:8.2f}" for dw in delta_w]) +
                    "".join([f"{w:8.2f}" for w in self.weights]) +
                    f"{self.bias:8.2f}"
                )
                print(row)

            if total_changes == 0:
                print(f"\nModel berhenti pada epoch ke {epoch}.")
                break
            epoch += 1

        return epoch

# 1. Input
alpha = float(input("Masukkan alpha : "))
theta = float(input("Masukkan theta : "))

# 2. Read dataset
file_path = "https://raw.githubusercontent.com/Zwelious/DSS_Tugas_Alfred/refs/heads/main/playtennis.csv"
df = pd.read_csv(file_path)

# 3. Binary:
df_binary = pd.get_dummies(df.drop('play', axis=1)).astype(int)
df_binary['play'] = df['play'].map({'yes': 1, 'no': -1})

# 4. Bipolar:
df_bipolar = df_binary.replace(0, -1)

X_binary = df_binary.drop('play', axis=1).values
y_binary = df_binary['play'].values

X_bipolar = df_bipolar.drop('play', axis=1).values
y_bipolar = df_bipolar['play'].values

# 6. Binary
print("\n Training dengan Binary :  ")
model_bin = Perceptron(num_features=X_binary.shape[1], alpha=alpha, theta=theta)
epochs_bin = model_bin.fit(X_binary, y_binary)

# 7. Bipolar
print("\n Training dengan Bipolar ===")
model_bip = Perceptron(num_features=X_bipolar.shape[1], alpha=alpha, theta=theta)
epochs_bip = model_bip.fit(X_bipolar, y_bipolar)

# 8. Conclusion1
print(f"\nTotal Epochs Binary   : {epochs_bin}")
print(f"Total Epochs Bipolar  : {epochs_bip}")


Masukkan alpha : 1
Masukkan theta : 1

 Training dengan Binary :  

Epoch 1
   X1   X2   X3   X4   X5   X6   X7   X8   X9       Yin    Yout  Target   Error    Δb     Δw1     Δw2     Δw3     Δw4     Δw5     Δw6     Δw7     Δw8     Δw9      w1      w2      w3      w4      w5      w6      w7      w8      w9    Bias
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    0    0    0    1    0    1    0    1    0      0.00      -1      -1       0  0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00
    1    0    0    1    0    1    0    1    0      0.00      -1      -1       0  0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0.00    0